In [1]:
import numpy as np
import matplotlib.pyplot as plt
from numba import cuda, float64, complex128
from numba.cuda import jit as cuda_jit
import math

import few

from few.trajectory.inspiral import EMRIInspiral
from few.trajectory.ode import KerrEccEqFlux
from few.amplitude.ampinterp2d import AmpInterpKerrEccEq
from few.summation.interpolatedmodesum import InterpolatedModeSum 


from few.utils.ylm import GetYlms

from few import get_file_manager

from few.waveform import GenerateEMRIWaveform, FastKerrEccentricEquatorialFlux

from few.utils.geodesic import get_fundamental_frequencies

from few.utils.constants import YRSID_SI
from smt.sampling_methods import LHS


import os
import sys

# Changing directory to FEWNEW/work
# to import stuffs
os.chdir('/nfs/home/svu/e1498138/localgit/FEWNEW/work/')
sys.path.insert(0, '/nfs/home/svu/e1498138/localgit/FEWNEW/work/')

import GWfuncs
import loglikebasic
import modeselector
import parismc
# import gc
import pickle
import cupy as cp

# tune few configuration
cfg_set = few.get_config_setter(reset=True)
cfg_set.set_log_level("info")

# GPU configuration 
use_gpu = True
force_backend = "cuda12x"  
dt = 10     # Time step
T = 0.25     # Total time

print('Initializing waveform generator...')
# keyword arguments for inspiral generator 
inspiral_kwargs={
        "func": 'KerrEccEqFlux',
        "DENSE_STEPPING": 0, #change to 1/True for uniform sampling
        "include_minus_m": False, 
}

# keyword arguments for inspiral generator 
amplitude_kwargs = {
    "force_backend": force_backend # Force GPU
}

# keyword arguments for Ylm generator (GetYlms)
Ylm_kwargs = {
    "force_backend": force_backend,  # Force GPU
    # "assume_positive_m": True  # if we assume positive m, it will generate negative m for all m>0
}

# keyword arguments for summation generator (InterpolatedModeSum)
sum_kwargs_comb = {
    "force_backend": force_backend,  # Force GPU
    "pad_output": True,
}

sum_kwargs_sep = {
    "force_backend": force_backend,  # Force GPU
    "pad_output": True,
    "separate_modes": True,
}

print("Creating GenerateEMRIWaveform class...")
# Kerr eccentric flux
waveform_gen_comb = GenerateEMRIWaveform(
    FastKerrEccentricEquatorialFlux, 
    frame='detector',
    inspiral_kwargs=inspiral_kwargs, 
    amplitude_kwargs=amplitude_kwargs, 
    Ylm_kwargs=Ylm_kwargs,
    sum_kwargs=sum_kwargs_comb,
    use_gpu=use_gpu
)

# Kerr eccentric flux
waveform_gen_sep = GenerateEMRIWaveform(
    FastKerrEccentricEquatorialFlux, 
    frame='detector',
    inspiral_kwargs=inspiral_kwargs, 
    amplitude_kwargs=amplitude_kwargs, 
    Ylm_kwargs=Ylm_kwargs,
    sum_kwargs=sum_kwargs_sep,
    use_gpu=use_gpu
)


print('Done initializing waveform generator.')

print("Creating GravWaveAnalysis class...")
gwf = GWfuncs.GravWaveAnalysis(T, dt)

print("Initializing loglike class...")
# Source parameters
m1 = 1e6
m2 = 3e1
a = 0.7
p0 = 7.5
e0 = 0.4 
## NOTE: BELOW THIS ALL ARE FIXED
xI0 = 1.0
dist = 0.5 # Gpc
# Polar and azimuthal angles .. detector frame
# S = Solar system barycenter
# K = spin angular momentum of the MBH
qS = 0.5 
phiS = 1 
qK = 1 #fixed
phiK = phiS + np.pi/3
# Phases
Phi_phi0 = 0.4
Phi_theta0 = 0.0 # equatorial
Phi_r0 = 0.5

params_star = (m1, m2, a, p0, e0, xI0, dist, qS, phiS, qK, phiK, Phi_phi0, Phi_theta0, Phi_r0)
param_true = [np.log10(m1), np.log10(m2), a, p0, e0]

# NOTE: change verbose argument for debugging
loglike_obj = loglikebasic.LogLike(params_star, waveform_gen_comb, gwf, M_init=5, verbose=False, waveform_gen_sep=waveform_gen_sep, noise_weighted=True)
print('Done initializing loglike class.')
print("Setting up log_density and prior functions...")
print('Calculating SNR...')
data = loglike_obj.signal
data_snr = gwf.rhostat(data)
print('SNR calculated:', data_snr)
print("Setting up log_density and prior functions...")

Initializing waveform generator...
Creating GenerateEMRIWaveform class...
Done initializing waveform generator.
Creating GravWaveAnalysis class...
Initializing loglike class...
Done initializing loglike class.
Setting up log_density and prior functions...
Calculating SNR...
SNR calculated: 107.43944631400305
Setting up log_density and prior functions...


In [2]:

def log_density(params):
    params = np.asarray(params)

    n_samples = params.shape[0] 
    log_likes = np.zeros(n_samples)


    for i in range(n_samples):
        logm1, logm2, a, p0, e0 = params[i]
        m1 = 10**logm1
        m2 = 10**logm2

        loglike = loglike_obj(np.array([m1, m2, a, p0, e0, xI0, dist, qS, phiS, qK, phiK, Phi_phi0, Phi_theta0, Phi_r0]))
        log_likes[i] = loglike

    return log_likes

def prior_transform(u):

    # # sampling_test => 30x fisher box
    logm1lim = [5.995531126784557, 6.004468873215443]
    logm2lim = [1.4746191268598186, 1.4796233825795062]
    alim = [0.6919479173260448, 0.7080520826739551]
    p0lim = [7.455582230927566, 7.544417769072434]
    e0lim = [0.3980771809772245, 0.40192281902277555]

    # box1 = 3 x box of sampling_test= 90x fisher box
    # logm1lim = [5.9865933803536695, 6.0134066196463305]
    # logm2lim = [1.469614871140131, 1.4846276382991939]
    # alim = [0.6758437519781343, 0.7241562480218656]
    # p0lim = [7.366746692782698, 7.633253307217302]
    # e0lim = [0.3942315429316735, 0.40576845706832654]

    transformed = np.zeros_like(u)

    # Uniform in log for masses

    # m1
    transformed[:, 0] = (logm1lim[1] - logm1lim[0]) * u[:, 0] + logm1lim[0]

    # m2
    transformed[:, 1] = (logm2lim[1] - logm2lim[0]) * u[:, 1] + logm2lim[0]

    # Linear in others 

    # a
    transformed[:, 2] = (alim[1] - alim[0]) * u[:, 2] + alim[0]

    # p0
    transformed[:, 3] = (p0lim[1] - p0lim[0]) * u[:, 3] + p0lim[0] 

    # e0
    transformed[:, 4] = (e0lim[1] - e0lim[0]) * u[:, 4] + e0lim[0]

    
    return transformed

In [3]:
import stableemrifisher
stableemrifisher.__file__
from tqdm import tqdm
from stableemrifisher.fisher.fisher import StableEMRIFisher


startup


In [4]:
waveform_class = FastKerrEccentricEquatorialFlux
waveform_class_kwargs = dict(inspiral_kwargs=inspiral_kwargs,
                             amplitude_kwargs=amplitude_kwargs,
                             Ylm_kwargs=Ylm_kwargs,
                             sum_kwargs=sum_kwargs_comb,
                             use_gpu=use_gpu)

 
#waveform generator setup
waveform_generator = GenerateEMRIWaveform
waveform_generator_kwargs = dict(frame='detector')

In [5]:
from lisatools.sensitivity import get_sensitivity, CornishLISASens

sef = StableEMRIFisher(waveform_class=waveform_class, 
                       waveform_class_kwargs=waveform_class_kwargs,
                       waveform_generator=waveform_generator,
                       waveform_generator_kwargs=waveform_generator_kwargs,
                      stats_for_nerds = True, use_gpu = use_gpu,
                      deriv_type='stable', noise_model=get_sensitivity, noise_kwargs={'sens_fn':CornishLISASens, 'return_type': 'PSD'}, channels=["A"])


# Step 1: Calc LHS samples

In [6]:
ndim=5
n_samples = int(10)
xlimits = np.column_stack([
        np.zeros(ndim, dtype=float),
        np.ones(ndim, dtype=float),
    ])
sampling = LHS(xlimits=xlimits)
lhs_points = np.clip(sampling(n_samples), 0.0, 1.0)

In [7]:
physical_points = prior_transform(lhs_points)

In [8]:
physical_points

array([[5.99776556, 1.47486934, 0.70563646, 7.46002401, 0.40019228],
       [5.99865934, 1.47787189, 0.69597396, 7.53109244, 0.39980772],
       [5.99955311, 1.47837232, 0.69758438, 7.53997599, 0.40173054],
       [5.99597801, 1.47637062, 0.70241562, 7.48667467, 0.39942315],
       [6.00134066, 1.47687104, 0.69436354, 7.52220888, 0.39903859],
       [6.00312821, 1.47937317, 0.69275313, 7.50444178, 0.40057685],
       [6.00044689, 1.47587019, 0.70724687, 7.46890756, 0.40134597],
       [6.00223444, 1.47536977, 0.70080521, 7.47779112, 0.39865403],
       [5.99687179, 1.47887274, 0.70402604, 7.49555822, 0.40096141],
       [6.00402199, 1.47737147, 0.69919479, 7.51332533, 0.39826946]])

In [9]:
trans_physical_points = physical_points.copy()

In [10]:
trans_physical_points[:,0] = 10**trans_physical_points[:,0]
trans_physical_points[:,1] = 10**trans_physical_points[:,1]

In [11]:
for point in trans_physical_points: 
    print(point)

[9.94868232e+05 2.98448458e+01 7.05636458e-01 7.46002401e+00
 4.00192282e-01]
[9.96917772e+05 3.00518971e+01 6.95973959e-01 7.53109244e+00
 3.99807718e-01]
[9.98971533e+05 3.00865450e+01 6.97584375e-01 7.53997599e+00
 4.01730537e-01]
[9.90781786e+05 2.99481925e+01 7.02415625e-01 7.48667467e+00
 3.99423154e-01]
[1.00309176e+06 2.99827209e+01 6.94363542e-01 7.52220888e+00
 3.99038590e-01]
[1.00722898e+06 3.01559608e+01 6.92753126e-01 7.50444178e+00
 4.00576846e-01]
[1.00102953e+06 2.99137039e+01 7.07246874e-01 7.46890756e+00
 4.01345973e-01]
[1.00515824e+06 2.98792550e+01 7.00805208e-01 7.47779112e+00
 3.98654027e-01]
[9.92822907e+05 3.01212329e+01 7.04026041e-01 7.49555822e+00
 4.00961410e-01]
[1.00930398e+06 3.00172891e+01 6.99194792e-01 7.51332533e+00
 3.98269463e-01]


# Step 2: Comp Fisher on LHS samples

In [12]:
der_order = 4
Ndelta = 8
stability_plot = False
param_names = ['m1','m2','a','p0','e0']


In [13]:
fixed_params = [xI0, dist, qS, phiS, qK, phiK, Phi_phi0, Phi_theta0, Phi_r0]

In [14]:
pars_list = [list(params) + fixed_params for params in trans_physical_points]
pars_list

[[994868.2323181228,
  29.84484581998772,
  0.7056364578717685,
  7.460024007834809,
  0.40019228190227757,
  1.0,
  0.5,
  0.5,
  1,
  1,
  2.0471975511965974,
  0.4,
  0.0,
  0.5],
 [996917.7715946073,
  30.051897097575107,
  0.6959739586630224,
  7.531092438350704,
  0.3998077180977225,
  1.0,
  0.5,
  0.5,
  1,
  1,
  2.0471975511965974,
  0.4,
  0.0,
  0.5],
 [998971.5331500926,
  30.08654502913166,
  0.6975843751978134,
  7.539975992165191,
  0.401730537120498,
  1.0,
  0.5,
  0.5,
  1,
  1,
  2.0471975511965974,
  0.4,
  0.0,
  0.5],
 [990781.7858980765,
  29.948192524345522,
  0.7024156248021866,
  7.48667466927827,
  0.3994231542931674,
  1.0,
  0.5,
  0.5,
  1,
  1,
  2.0471975511965974,
  0.4,
  0.0,
  0.5],
 [1003091.7579094437,
  29.98272089110576,
  0.6943635421282314,
  7.522208884536218,
  0.39903859048861223,
  1.0,
  0.5,
  0.5,
  1,
  1,
  2.0471975511965974,
  0.4,
  0.0,
  0.5],
 [1007228.9763983479,
  30.155960778904436,
  0.6927531255934403,
  7.504441776907243,


In [15]:
Fishers = []
for params in pars_list: 
    # point corresponds to pars_list
    Fisher = sef(*params, param_names = param_names, 
             T = T, dt = dt, 
             der_order = der_order, 
             Ndelta = Ndelta, 
             stability_plot = stability_plot,
            #  delta_range = delta_range,
            live_dangerously = False)
    Fishers.append(Fisher)

T:  0.25 dt:  10
Body is not plunging, Fisher should be stable.
wave ndim: 2
Computing SNR for parameters: (994868.2323181228, 29.84484581998772, 0.7056364578717685, 7.460024007834809, 0.40019228190227757, 1.0, 0.5, 0.5, 1, 1, 2.0471975511965974, 0.4, 0.0, 0.5)
Waveform Generated. SNR: 109.42107558590332
calculating stable deltas...
Gamma_ii for m1: 661.0137550374918
Gamma_ii for m1: 661.0137550244563
Gamma_ii for m1: 661.0137550192164
Gamma_ii for m1: 661.0137550852799
Gamma_ii for m1: 661.0137555213818
Gamma_ii for m1: 661.0137511457865
Gamma_ii for m1: 661.0137474013409
Gamma_ii for m1: 661.0137890019711
[1.9720558139298698e-11, 7.926955091028257e-12, 9.994258196867323e-11, 6.597471082099871e-10, 6.619522301711262e-09, 5.664701582749123e-09, 6.29345875830311e-08]
1
Gamma_ii for m2: 25524649344.619034
Gamma_ii for m2: 25524649343.903046
Gamma_ii for m2: 25524649343.49321
Gamma_ii for m2: 25524649352.80348
Gamma_ii for m2: 25524649379.573982
Gamma_ii for m2: 25524649314.730515
Gamma_i

In [16]:
J_new = np.eye(5)
J_new[0, 0] = m1 * np.log(10) 
J_new[1, 1] = m2 * np.log(10) 

In [17]:
Fishers_scaled = []

In [18]:
for Fisher in Fishers:
    Fishers_scaled.append(J_new.T @ Fisher @ J_new)

In [19]:
with open('calc_fisher_lhs.pkl', 'wb') as f:
    pickle.dump((lhs_points, Fishers_scaled), f)


# Step 3: Optimize around LHS points

In [20]:
print('Setting up ParisMC sampler...')
config = parismc.SamplerConfig(
    gamma=50,                    # Covariance update frequency 
)

Setting up ParisMC sampler...


In [21]:
with open('calc_fisher_lhs.pkl', 'rb') as f:
    lhs_points, Fishers = pickle.load(f)

In [22]:
cov_matrices = [np.linalg.inv(0.01*Fishers[i]) for i in range(len(Fishers))]

In [23]:
ndim = 5
n_seed = 10

In [24]:
init_cov_list = [cov_matrices[i] for i in range(n_seed)]
init_cov_list

[array([[ 2.75491334e-07,  1.56222779e-07,  4.92575765e-07,
         -2.72544042e-06, -1.20346238e-07],
        [ 1.56222779e-07,  9.00513263e-08,  2.78493053e-07,
         -1.53927023e-06, -6.94338827e-08],
        [ 4.92575765e-07,  2.78493053e-07,  8.81302139e-07,
         -4.87683268e-06, -2.14446800e-07],
        [-2.72544042e-06, -1.53927023e-06, -4.87683268e-06,
          2.69899893e-05,  1.18537904e-06],
        [-1.20346238e-07, -6.94338827e-08, -2.14446800e-07,
          1.18537904e-06,  5.35752170e-08]]),
 array([[ 3.75394408e-07,  2.07149728e-07,  6.83116188e-07,
         -3.76083706e-06, -1.59370392e-07],
        [ 2.07149728e-07,  1.16377176e-07,  3.75749583e-07,
         -2.06653174e-06, -8.95842714e-08],
        [ 6.83116188e-07,  3.75749583e-07,  1.24392170e-06,
         -6.84908304e-06, -2.88990184e-07],
        [-3.76083706e-06, -2.06653174e-06, -6.84908304e-06,
          3.77151886e-05,  1.58952423e-06],
        [-1.59370392e-07, -8.95842714e-08, -2.88990184e-07,
  

In [25]:
sampler = parismc.Sampler(
    ndim=ndim, 
    n_seed=n_seed,
    log_density_func=log_density,
    init_cov_list=init_cov_list,
    prior_transform=prior_transform,
    config=config
)

In [26]:
physical_points = prior_transform(lhs_points)

In [27]:
physical_points

array([[5.99776556, 1.47486934, 0.70563646, 7.46002401, 0.40019228],
       [5.99865934, 1.47787189, 0.69597396, 7.53109244, 0.39980772],
       [5.99955311, 1.47837232, 0.69758438, 7.53997599, 0.40173054],
       [5.99597801, 1.47637062, 0.70241562, 7.48667467, 0.39942315],
       [6.00134066, 1.47687104, 0.69436354, 7.52220888, 0.39903859],
       [6.00312821, 1.47937317, 0.69275313, 7.50444178, 0.40057685],
       [6.00044689, 1.47587019, 0.70724687, 7.46890756, 0.40134597],
       [6.00223444, 1.47536977, 0.70080521, 7.47779112, 0.39865403],
       [5.99687179, 1.47887274, 0.70402604, 7.49555822, 0.40096141],
       [6.00402199, 1.47737147, 0.69919479, 7.51332533, 0.39826946]])

In [28]:
lhs_logden = log_density(physical_points)
lhs_logden

array([-7.65985612e-07, -6.27086394e-07,  9.19910061e-07,  3.38614011e-06,
        8.66228852e-07, -1.96837045e-07, -5.97037053e-07, -1.04929729e-05,
       -1.63907083e-06,  4.73819245e-07])

In [29]:
point_blocks = []
point_blocks.append(lhs_points)
log_blocks = []
log_blocks.append(np.asarray(lhs_logden, dtype=float).reshape(-1))

external_lhs_points = np.vstack(point_blocks) 
external_lhs_log_densities = np.concatenate(log_blocks)

In [30]:
print('Running sampling...')
sampler.run_sampling(
    num_iterations=int(50), 
    savepath='./lhs_try/',
    print_iter=10, # Print progress every n iterations
    external_lhs_points=external_lhs_points,
    external_lhs_log_densities=external_lhs_log_densities
)

Running sampling...


Sampling:   0%|          | 0/49 [00:00<?, ?it/s]

/nfs/home/svu/e1498138/localgit/parismc/parismc/sampler.py:765: RuntimeWarning: divide by zero encountered in log
  logZ = c_term - np.log(Nsum) + np.log(wsum)
/nfs/home/svu/e1498138/localgit/parismc/parismc/sampler.py:765: RuntimeWarning: invalid value encountered in scalar add
  logZ = c_term - np.log(Nsum) + np.log(wsum)


In [31]:
print("Extracting results...")
samples, weights = sampler.get_samples_with_weights(flatten=True)


Extracting results...


In [32]:
np.max(sampler.searched_log_densities_list)

0.00013427362680604584

In [33]:
np.argmax(sampler.searched_log_densities_list)

22

In [34]:
max_point_iter1 = sampler.searched_points_list[0][np.argmax(sampler.searched_log_densities_list)]    
max_point_iter1

array([0.75043088, 0.15016393, 0.55080171, 0.24546171, 0.14986972])

In [35]:
max_physical_point_iter1 = prior_transform(np.array([max_point_iter1]))
max_physical_point_iter1

array([[6.00223829, 1.47537059, 0.70081812, 7.47738795, 0.39865353]])

# Step 4: Next LHS sampling

Calc Fisher

In [36]:
pars_list = [list(param) + fixed_params for param in max_physical_point_iter1]
pars_list

[[6.002238287671177,
  1.475370585582307,
  0.7008181191984197,
  7.4773879543374395,
  0.3986535256895246,
  1.0,
  0.5,
  0.5,
  1,
  1,
  2.0471975511965974,
  0.4,
  0.0,
  0.5]]

In [37]:
params_new = pars_list[0].copy()
params_new[0] = 10**params_new[0]
params_new[1] = 10**params_new[1]
params_new 

[1005167.1517448274,
 29.879311475297246,
 0.7008181191984197,
 7.4773879543374395,
 0.3986535256895246,
 1.0,
 0.5,
 0.5,
 1,
 1,
 2.0471975511965974,
 0.4,
 0.0,
 0.5]

In [38]:
Fisher_new = sef(*params_new, param_names = param_names, 
            T = T, dt = dt, 
            der_order = der_order, 
            Ndelta = Ndelta, 
            stability_plot = stability_plot,
        #  delta_range = delta_range,
        live_dangerously = False)


T:  0.25 dt:  10
Body is not plunging, Fisher should be stable.
wave ndim: 2
Computing SNR for parameters: (1005167.1517448274, 29.879311475297246, 0.7008181191984197, 7.4773879543374395, 0.3986535256895246, 1.0, 0.5, 0.5, 1, 1, 2.0471975511965974, 0.4, 0.0, 0.5)
Waveform Generated. SNR: 107.16950726066709
calculating stable deltas...
Gamma_ii for m1: 579.7646662865438
Gamma_ii for m1: 579.7646662722536
Gamma_ii for m1: 579.7646662802138
Gamma_ii for m1: 579.7646661565777
Gamma_ii for m1: 579.7646659414961
Gamma_ii for m1: 579.7646627978877
Gamma_ii for m1: 579.7646516338413
Gamma_ii for m1: 579.7646454372191
[2.464829086569602e-11, 1.372992434096492e-11, 2.1325209150396034e-10, 3.70980916585902e-10, 5.422214584202255e-09, 1.9256169567460917e-08, 1.0688168406381151e-08]
1
Gamma_ii for m2: 20953640199.571507
Gamma_ii for m2: 20953640199.60327
Gamma_ii for m2: 20953640200.06711
Gamma_ii for m2: 20953640202.80276
Gamma_ii for m2: 20953640345.66521
Gamma_ii for m2: 20953640251.116478
Gamma

In [39]:
Fisher_new_scaled = J_new.T @ Fisher_new @ J_new

In [40]:
cov_new = np.linalg.inv(Fisher_new_scaled)

In [41]:
sigmas_new = np.sqrt(np.diag(cov_new))
sigmas_new

array([6.03823241e-05, 3.36326707e-05, 1.08001645e-04, 5.95589714e-04,
       2.59832624e-05])

In [42]:
max_physical_point_iter1

array([[6.00223829, 1.47537059, 0.70081812, 7.47738795, 0.39865353]])

In [43]:
for i, val in enumerate(max_physical_point_iter1[0]):
    # fisher box = 2.5 * sigma
    # 30x fisher box = 75*sigma
    prior_low = val - 0.7*75*sigmas_new[i]
    prior_high = val + 0.7*75*sigmas_new[i]
    
    print(f"0.7*30x-sigma prior range: [{prior_low:.10e}, {prior_high:.10e}]")

0.7*30x-sigma prior range: [5.9990682157e+00, 6.0054083597e+00]
0.7*30x-sigma prior range: [1.4736048704e+00, 1.4771363008e+00]
0.7*30x-sigma prior range: [6.9514803285e-01, 7.0648820555e-01]
0.7*30x-sigma prior range: [7.4461194943e+00, 7.5086564143e+00]
0.7*30x-sigma prior range: [3.9728940441e-01, 4.0001764696e-01]


Compute LHS points

In [44]:
sampling = LHS(xlimits=xlimits)
lhs_points_new = np.clip(sampling(n_samples), 0.0, 1.0)
lhs_points_new

array([[0.95, 0.95, 0.45, 0.95, 0.25],
       [0.15, 0.65, 0.25, 0.55, 0.45],
       [0.45, 0.75, 0.15, 0.65, 0.55],
       [0.05, 0.45, 0.55, 0.75, 0.15],
       [0.35, 0.05, 0.35, 0.15, 0.75],
       [0.85, 0.25, 0.75, 0.35, 0.35],
       [0.55, 0.15, 0.65, 0.85, 0.05],
       [0.75, 0.85, 0.95, 0.05, 0.85],
       [0.65, 0.35, 0.05, 0.25, 0.95],
       [0.25, 0.55, 0.85, 0.45, 0.65]])

In [45]:
def prior_transform_new(u):
    # TODO: when changign to py file pls find good way to make new funcs 
    # # # sampling_test => 30x fisher box
    # logm1lim = [5.995531126784557, 6.004468873215443]
    # logm2lim = [1.4746191268598186, 1.4796233825795062]
    # alim = [0.6919479173260448, 0.7080520826739551]
    # p0lim = [7.455582230927566, 7.544417769072434]
    # e0lim = [0.3980771809772245, 0.40192281902277555]

    # SECOND ITER
    logm1lim = [5.9990682157e+00, 6.0054083597e+00]
    logm2lim = [1.4736048704e+00, 1.4771363008e+00]
    alim = [6.9514803285e-01, 7.0648820555e-01]
    p0lim = [7.4461194943e+00, 7.5086564143e+00]
    e0lim = [3.9728940441e-01, 4.0001764696e-01]

    transformed = np.zeros_like(u)

    # Uniform in log for masses

    # m1
    transformed[:, 0] = (logm1lim[1] - logm1lim[0]) * u[:, 0] + logm1lim[0]

    # m2
    transformed[:, 1] = (logm2lim[1] - logm2lim[0]) * u[:, 1] + logm2lim[0]

    # Linear in others 

    # a
    transformed[:, 2] = (alim[1] - alim[0]) * u[:, 2] + alim[0]

    # p0
    transformed[:, 3] = (p0lim[1] - p0lim[0]) * u[:, 3] + p0lim[0] 

    # e0
    transformed[:, 4] = (e0lim[1] - e0lim[0]) * u[:, 4] + e0lim[0]

    
    return transformed

In [46]:
physical_points_new = prior_transform_new(lhs_points_new)
physical_points_new

array([[6.00509135, 1.47695973, 0.70025111, 7.50552957, 0.39797147],
       [6.00001924, 1.4759003 , 0.69798308, 7.4805148 , 0.39851711],
       [6.00192128, 1.47625344, 0.69684906, 7.48676849, 0.39878994],
       [5.99938522, 1.47519401, 0.70138513, 7.49302218, 0.39769864],
       [6.00128727, 1.47378144, 0.69911709, 7.45550003, 0.39933559],
       [6.00445734, 1.47448773, 0.70365316, 7.46800742, 0.39824429],
       [6.00255529, 1.47413458, 0.70251915, 7.49927588, 0.39742582],
       [6.00382332, 1.47660659, 0.7059212 , 7.44924634, 0.39960841],
       [6.00318931, 1.47484087, 0.69571504, 7.46175372, 0.39988123],
       [6.00065325, 1.47554716, 0.70478718, 7.47426111, 0.39906276]])

In [47]:
trans_physical_points_new = physical_points_new.copy()
trans_physical_points_new[:,0] = 10**trans_physical_points_new[:,0]
trans_physical_points_new[:,1] = 10**trans_physical_points_new[:,1]

In [48]:
trans_physical_points_new

array([[1.01179226e+06, 2.99888443e+01, 7.00251111e-01, 7.50552957e+00,
        3.97971465e-01],
       [1.00004430e+06, 2.99157779e+01, 6.97983076e-01, 7.48051480e+00,
        3.98517114e-01],
       [1.00443371e+06, 2.99401136e+01, 6.96849059e-01, 7.48676849e+00,
        3.98789938e-01],
       [9.98585425e+05, 2.98671659e+01, 7.01385128e-01, 7.49302218e+00,
        3.97698641e-01],
       [1.00296844e+06, 2.97701787e+01, 6.99117093e-01, 7.45550003e+00,
        3.99335586e-01],
       [1.01031625e+06, 2.98186329e+01, 7.03653162e-01, 7.46800742e+00,
        3.98244289e-01],
       [1.00590113e+06, 2.97943959e+01, 7.02519145e-01, 7.49927588e+00,
        3.97425817e-01],
       [1.00884239e+06, 2.99644690e+01, 7.05921197e-01, 7.44924634e+00,
        3.99608411e-01],
       [1.00737069e+06, 2.98428895e+01, 6.95715041e-01, 7.46175372e+00,
        3.99881235e-01],
       [1.00150530e+06, 2.98914620e+01, 7.04787180e-01, 7.47426111e+00,
        3.99062762e-01]])

In [49]:
pars_list = [list(params) + fixed_params for params in trans_physical_points_new]
pars_list

[[1011792.2592476187,
  29.988844292592624,
  0.700251110565,
  7.5055295683,
  0.3979714650475,
  1.0,
  0.5,
  0.5,
  1,
  1,
  2.0471975511965974,
  0.4,
  0.0,
  0.5],
 [1000044.2965012706,
  29.915777891410073,
  0.6979830760250001,
  7.4805148003,
  0.39851711355749997,
  1.0,
  0.5,
  0.5,
  1,
  1,
  2.0471975511965974,
  0.4,
  0.0,
  0.5],
 [1004433.7117827733,
  29.94011355670633,
  0.696849058755,
  7.4867684923,
  0.3987899378125,
  1.0,
  0.5,
  0.5,
  1,
  1,
  2.0471975511965974,
  0.4,
  0.0,
  0.5],
 [998585.4248699596,
  29.867165885657673,
  0.7013851278350001,
  7.4930221843,
  0.3976986407925,
  1.0,
  0.5,
  0.5,
  1,
  1,
  2.0471975511965974,
  0.4,
  0.0,
  0.5],
 [1002968.4368416741,
  29.77017872379601,
  0.699117093295,
  7.455500032300001,
  0.3993355863225,
  1.0,
  0.5,
  0.5,
  1,
  1,
  2.0471975511965974,
  0.4,
  0.0,
  0.5],
 [1010316.2496058852,
  29.818632872573016,
  0.703653162375,
  7.4680074163,
  0.3982442893025,
  1.0,
  0.5,
  0.5,
  1,
  1

In [50]:
Fishers_new = []
for params in pars_list: 
    # point corresponds to pars_list
    Fisher = sef(*params, param_names = param_names, 
             T = T, dt = dt, 
             der_order = der_order, 
             Ndelta = Ndelta, 
             stability_plot = stability_plot,
            #  delta_range = delta_range,
            live_dangerously = False)
    Fishers_new.append(Fisher)

T:  0.25 dt:  10
Body is not plunging, Fisher should be stable.
wave ndim: 2
Computing SNR for parameters: (1011792.2592476187, 29.988844292592624, 0.700251110565, 7.5055295683, 0.3979714650475, 1.0, 0.5, 0.5, 1, 1, 2.0471975511965974, 0.4, 0.0, 0.5)
Waveform Generated. SNR: 104.99995787010317
calculating stable deltas...
Gamma_ii for m1: 514.7898085664492
Gamma_ii for m1: 514.7898085714903
Gamma_ii for m1: 514.7898085203099
Gamma_ii for m1: 514.7898085495493
Gamma_ii for m1: 514.7898088828637
Gamma_ii for m1: 514.7898100553491
Gamma_ii for m1: 514.7897973685007
Gamma_ii for m1: 514.7897226113315
[9.79254381946807e-12, 9.942009195428675e-11, 5.6798830063321364e-11, 6.474765837209708e-10, 2.2776003281159354e-09, 2.4644716155328136e-08, 1.4521884543429102e-07]
0
Gamma_ii for m2: 17190526600.70108
Gamma_ii for m2: 17190526599.799824
Gamma_ii for m2: 17190526605.999405
Gamma_ii for m2: 17190526572.9114
Gamma_ii for m2: 17190526520.85954
Gamma_ii for m2: 17190526409.258137
Gamma_ii for m2: 

In [51]:
Fishers_scaled_new = []
for Fisher in Fishers_new:
    Fishers_scaled_new.append(J_new.T @ Fisher @ J_new)

In [52]:
cov_matrices_new = [np.linalg.inv(0.01*Fishers_scaled_new[i]) for i in range(len(Fishers_scaled_new))]

In [53]:
init_cov_list_new = [cov_matrices_new[i] for i in range(n_seed)]
init_cov_list_new

[array([[ 4.79607624e-07,  2.58717194e-07,  8.58802523e-07,
         -4.73414814e-06, -1.99357532e-07],
        [ 2.58717194e-07,  1.42199925e-07,  4.61708206e-07,
         -2.54264864e-06, -1.09629213e-07],
        [ 8.58802523e-07,  4.61708206e-07,  1.53887129e-06,
         -8.48399545e-06, -3.55671542e-07],
        [-4.73414814e-06, -2.54264864e-06, -8.48399545e-06,
          4.67776718e-05,  1.95886849e-06],
        [-1.99357532e-07, -1.09629213e-07, -3.55671542e-07,
          1.95886849e-06,  8.45645554e-08]]),
 array([[ 3.13918347e-07,  1.75591841e-07,  5.64090833e-07,
         -3.10590858e-06, -1.35770719e-07],
        [ 1.75591841e-07,  9.98827602e-08,  3.14570528e-07,
         -1.73023460e-06, -7.72921470e-08],
        [ 5.64090833e-07,  3.14570528e-07,  1.01430058e-06,
         -5.58541424e-06, -2.43138059e-07],
        [-3.10590858e-06, -1.73023460e-06, -5.58541424e-06,
          3.07604291e-05,  1.33745052e-06],
        [-1.35770719e-07, -7.72921470e-08, -2.43138059e-07,
  

In [54]:
sampler = parismc.Sampler(
    ndim=ndim, 
    n_seed=n_seed,
    log_density_func=log_density,
    init_cov_list=init_cov_list_new,
    prior_transform=prior_transform_new,
    config=config
)

In [55]:
physical_points_new = prior_transform_new(lhs_points_new)

In [56]:
lhs_logden_new = log_density(physical_points_new)
lhs_logden_new

array([-1.17814022e-06,  1.07595515e-06,  2.13965985e-04,  3.45916163e-07,
        6.19809062e-07,  8.86640405e-07, -3.06060660e-07,  2.43946701e-07,
        4.77161310e-06, -2.03723362e-07])

In [57]:
point_blocks_new = []
point_blocks_new.append(lhs_points_new)
log_blocks_new = []
log_blocks_new.append(np.asarray(lhs_logden_new, dtype=float).reshape(-1))

external_lhs_points_new = np.vstack(point_blocks_new) 
external_lhs_log_densities_new = np.concatenate(log_blocks_new)

In [58]:
print('Running sampling...')
sampler.run_sampling(
    num_iterations=int(50), 
    savepath='./lhs_try/',
    print_iter=10, # Print progress every n iterations
    external_lhs_points=external_lhs_points_new,
    external_lhs_log_densities=external_lhs_log_densities_new
)

Running sampling...


Sampling:   0%|          | 0/49 [00:00<?, ?it/s]

/nfs/home/svu/e1498138/localgit/parismc/parismc/sampler.py:765: RuntimeWarning: divide by zero encountered in log
  logZ = c_term - np.log(Nsum) + np.log(wsum)
/nfs/home/svu/e1498138/localgit/parismc/parismc/sampler.py:765: RuntimeWarning: invalid value encountered in scalar add
  logZ = c_term - np.log(Nsum) + np.log(wsum)


In [59]:
print("Extracting results...")
samples, weights = sampler.get_samples_with_weights(flatten=True)


Extracting results...


In [60]:
np.max(sampler.searched_log_densities_list)

0.000673460111501866

In [61]:
max_physical_point_iter2 = prior_transform_new(np.array([sampler.searched_points_list[0][np.argmax(sampler.searched_log_densities_list)]]))
max_physical_point_iter2

array([[6.00193835, 1.4762591 , 0.69690276, 7.48514459, 0.39878658]])

In [62]:
param_true

[6.0, 1.4771212547196624, 0.7, 7.5, 0.4]

In [63]:
math.dist(param_true, max_physical_point_iter1[0])

0.02284429240017612

In [64]:
math.dist(param_true, max_physical_point_iter2[0])

0.015370395335169491